In [5]:
import requests
import pandas as pd
import time

# GitHub API 인증 토큰 및 헤더 (실제 유효한 토큰으로 교체 필요)
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN" # 사용자 토큰을 사용합니다.
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}

# 검색을 위한 기본 필터 조건
# 🚨 주의: pushed:2022-01-01..2024-12-31 필터는 현재 2025년 5월 기준으로,
# 2025년에 업데이트된 저장소를 제외합니다. 이로 인해 결과가 0으로 나올 수 있습니다.
# 필요시 이 필터를 조정하세요. (예: pushed:>=2022-01-01 또는 pushed:2022-01-01..{오늘날짜})
BASE_FILTERS = "in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31"

# # 검색할 키워드 목록 정의 (키워드 자체에 큰따옴표 제거)
# plain_keywords_no_quotes = [

#     '"artificial intelligence" OR "computer vision" OR "deep learning" OR "machine learning" OR "image recognition"',
#     '"natural language processing" OR "nlp" OR "language model" OR "speech recognition"',
# ]

ai_adjectives = [
    "agent", "causal", "explainable", "generative", "multi modal",
    "quantum", "reliable", "responsible", "robotics", "symbolic"
]

combined_ai_keywords_no_quotes = []
for adj in ai_adjectives:
    combined_ai_keywords_no_quotes.append(f'"{adj}" AND "AI"') # 예: agentic AI (따옴표 없음)

keywords_list_for_query = []
# for kw in plain_keywords_no_quotes:
#     keywords_list_for_query.append([kw])
for kw in combined_ai_keywords_no_quotes:
    keywords_list_for_query.append([kw])

results_summary = []
total_repos_aggregated = 0 # 각 쿼리 결과의 총합을 저장할 변수

print(f"총 {len(keywords_list_for_query)}개의 키워드 쿼리를 실행합니다.")
print(f"사용될 기본 필터: {BASE_FILTERS}\n")
print("참고: 키워드 내 큰따옴표가 제거되어, 여러 단어 키워드는 AND 조건으로 검색됩니다.")
print("(예: 'artificial intelligence'는 'artificial' AND 'intelligence'로 검색)\n")

for i, group in enumerate(keywords_list_for_query):
    keyword_to_search = group[0]
    full_query = f"{keyword_to_search} {BASE_FILTERS}"
    
    url = "https://api.github.com/search/repositories"
    params = {
        "q": full_query,
        "per_page": 1
    }

    print(f"({i+1}/{len(keywords_list_for_query)}) 실행 중인 쿼리: {full_query}")

    try:
        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()
        count = data.get("total_count", 0)
        
        print(f"  → 결과: {count}개 저장소")
        
        if isinstance(count, int): # count가 정수일 때만 합산
            total_repos_aggregated += count
        
        repo_dict_entry = {keyword_to_search: count}
        results_summary.append(repo_dict_entry)

    except requests.exceptions.HTTPError as http_err:
        error_message = f"HTTP 오류: {http_err}"
        try:
            error_data = response.json()
            error_message += f" - {error_data.get('message', '')}"
            if 'errors' in error_data:
                for err_detail in error_data['errors']:
                    error_message += f" ({err_detail.get('resource', '')} {err_detail.get('field', '')} {err_detail.get('code', '')})"
        except ValueError:
            error_message += f" (응답 내용: {response.text})"
        print(f"  → ❌ {error_message}")
        results_summary.append({keyword_to_search: f"오류 ({response.status_code if response else 'N/A'})"})

    except requests.exceptions.RequestException as req_err:
        print(f"  → ❌ 요청 오류: {req_err}")
        results_summary.append({keyword_to_search: "요청 오류"})
        
    except Exception as e:
        print(f"  → ❌ 알 수 없는 오류: {e}")
        results_summary.append({keyword_to_search: "알 수 없는 오류"})

    time.sleep(3)

print(f"\n📦 모든 쿼리 실행 완료.")
# --- 총 합계 출력 ---
print(f"📊 전체 추정 합계 (각 쿼리 결과의 단순 합): {total_repos_aggregated}개")
print("   (참고: 이 합계는 여러 키워드에 해당하는 저장소가 중복 계산될 수 있습니다.)")


if results_summary:
    processed_results_for_df = []
    for entry in results_summary:
        for keyword, count_or_error in entry.items():
            processed_results_for_df.append({'Keyword': keyword, 'Repository Count': count_or_error})
    df_results = pd.DataFrame(processed_results_for_df)
    
    print("\n--- 개별 키워드 검색 결과 요약 ---")
    print(df_results.to_string())
else:
    print("\n결과가 없습니다.")

총 10개의 키워드 쿼리를 실행합니다.
사용될 기본 필터: in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31

참고: 키워드 내 큰따옴표가 제거되어, 여러 단어 키워드는 AND 조건으로 검색됩니다.
(예: 'artificial intelligence'는 'artificial' AND 'intelligence'로 검색)

(1/10) 실행 중인 쿼리: "agent" AND "AI" in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31
  → 결과: 2095개 저장소
(2/10) 실행 중인 쿼리: "causal" AND "AI" in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31
  → 결과: 172개 저장소
(3/10) 실행 중인 쿼리: "explainable" AND "AI" in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31
  → 결과: 64개 저장소
(4/10) 실행 중인 쿼리: "generative" AND "AI" in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31
  → 결과: 1079개 저장소
(5/10) 실행 중인 쿼리: "multi modal" AND "AI" in:name,description,r

KeyboardInterrupt: 

In [7]:
import requests
import pandas as pd
import time


GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}
author_keywords_series = pd.read_csv("keywords_2020~2024.csv")
author_keywords_series = author_keywords_series['keyword']

# keywords_list = []
# for kw in author_keywords_series:
#     keywords_list.append([kw])
    
keywords_list = [
    # ["machine learning"],
    # ["deep learning"],
    # ["artificial intelligence"],
    # ["natural language processing"],
    # ["computer vision"],
    # ["ai AND explainable"],
    # ["ai AND responsible"],
    # ["ai AND quantum"],
    # ["ai AND agentic"],
    ['ai'],
    ['nlp']
]

base = "in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31"

result = []
total_repos = 0
for group in keywords_list:
    or_query = " OR ".join(group)
    full_query = f"{or_query} {base}"
    
    url = "https://api.github.com/search/repositories"
    params = {
        "q": full_query,
        "per_page": 1  # just to get total_count
    }

    response = requests.get(url, headers=HEADERS, params=params)
    data = response.json()

    if response.status_code == 200:
        count = data["total_count"]
        total_repos += count
        print(f"쿼리 {full_query} → {count}개")
        repo_dic ={}
        repo_dic[group[0]] = count
        result.append(repo_dic)
    else:
        print("❌ 오류 발생:", data)
    time.sleep(3)

print(f"\n📦 전체 추정 합계: {total_repos}개")


쿼리 ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 → 14336개


KeyboardInterrupt: 

In [10]:
import requests
import pandas as pd
import time


GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}
author_keywords_series = pd.read_csv("keywords_2020~2024.csv")
author_keywords_series = author_keywords_series['keyword']

# keywords_list = []
# for kw in author_keywords_series:
#     keywords_list.append([kw])
    
keywords_list = [
    #['"multi modal" OR "multimodal" OR "multi-modal"' ],
    ['"multi-model" AND "AI"'],
    ['multi modal AND AI'],
    ['"multimodal" AND "AI"']
   
]

base = "in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31"

result = []
total_repos = 0
for group in keywords_list:
    or_query = " OR ".join(group)
    full_query = f"{or_query} {base}"
    
    url = "https://api.github.com/search/repositories"
    params = {
        "q": full_query,
        "per_page": 1  # just to get total_count
    }

    response = requests.get(url, headers=HEADERS, params=params)
    data = response.json()

    if response.status_code == 200:
        count = data["total_count"]
        total_repos += count
        print(f"쿼리 {full_query} → {count}개")
        repo_dic ={}
        repo_dic[group[0]] = count
        result.append(repo_dic)
    else:
        print("❌ 오류 발생:", data)
    time.sleep(3)

print(f"\n📦 전체 추정 합계: {total_repos}개")


쿼리 "multi-model" AND "AI" in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31 → 54개
쿼리 "multi modal" AND "AI" in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31 → 262개
쿼리 "multimodal" AND "AI" in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31 → 465개

📦 전체 추정 합계: 781개


In [116]:
import requests
import pandas as pd
import time


GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}
author_keywords_series = pd.read_csv("keywords_2020~2024.csv")
author_keywords_series = author_keywords_series['keyword']

# keywords_list = []
# for kw in author_keywords_series:
#     keywords_list.append([kw])
    
keywords_list = [
#     ["machine learning"],
#     ["deep learning"],
#     ["artificial intelligence"],
#     ["natural language processing"],
#     ["computer vision"],
    # ["ai AND explainable"],
    # ["ai AND responsible"],
    # ["ai AND quantum"],
    # ["ai AND agentic"],
    ['natural language processing']
]

base = "in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31"

result = []
total_repos = 0
for group in keywords_list:
    or_query = " OR ".join(group)
    full_query = f"({or_query}) {base}"
    
    url = "https://api.github.com/search/repositories"
    params = {
        "q": full_query,
        "per_page": 1  # just to get total_count
    }

    response = requests.get(url, headers=HEADERS, params=params)
    data = response.json()

    if response.status_code == 200:
        count = data["total_count"]
        total_repos += count
        print(f"쿼리 {full_query} → {count}개")
        repo_dic ={}
        repo_dic[group[0]] = count
        result.append(repo_dic)
    else:
        print("❌ 오류 발생:", data)
    time.sleep(3)

print(f"\n📦 전체 추정 합계: {total_repos}개")


쿼리 (natural language processing) in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 → 194개

📦 전체 추정 합계: 194개


In [121]:
import requests
import pandas as pd
import time


GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}
author_keywords_series = pd.read_csv("keywords_2020~2024.csv")
author_keywords_series = author_keywords_series['keyword']

# keywords_list = []
# for kw in author_keywords_series:
#     keywords_list.append([kw])
    
keywords_list = [
#     ["machine learning"],
#     ["deep learning"],
#     ["artificial intelligence"],
#     ["natural language processing"],
#     ["computer vision"],
    # ["ai AND explainable"],
    # ["ai AND responsible"],
    # ["ai AND quantum"],
    # ["ai AND agentic"],
    ['"machine learning"','"deep learning"','"artificial intelligence"','"ai"'],
    ['"natural language processing"','"language model"','"nlp"'],
    ['computer vision','cv']
]

base = "in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31"

result = []
total_repos = 0
for group in keywords_list:
    or_query = " OR ".join(group)
    full_query = f"{or_query} {base}"
    
    url = "https://api.github.com/search/repositories"
    params = {
        "q": full_query,
        "per_page": 1  # just to get total_count
    }

    response = requests.get(url, headers=HEADERS, params=params)
    data = response.json()

    if response.status_code == 200:
        count = data["total_count"]
        total_repos += count
        print(f"쿼리 {full_query} → {count}개")
        repo_dic ={}
        repo_dic[group[0]] = count
        result.append(repo_dic)
    else:
        print("❌ 오류 발생:", data)
    time.sleep(3)

print(f"\n📦 전체 추정 합계: {total_repos}개")


쿼리 machine learning OR deep learning OR artificial intelligence OR ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 → 21693개
쿼리 natural language processing OR language model OR nlp in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 → 4183개
쿼리 computer vision OR cv in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 → 17971개

📦 전체 추정 합계: 43847개


In [140]:
import requests
import pandas as pd
import time


GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}
author_keywords_series = pd.read_csv("keywords_2020~2024.csv")
author_keywords_series = author_keywords_series['keyword']

# keywords_list = []
# for kw in author_keywords_series:
#     keywords_list.append([kw])
    
keywords_list = [

    # ['"machine learning"','"deep learning"','"artificial intelligence"'],
    # ['"natural language processing"','"language model"','"nlp"'],
    # ['"computer vision"','"multi-modal"','"robotics"'],
    ['"ai"','"agentic"'],
    #['"ai"','"quantum"'],
    #['"ai"','"explainable"'],
    
    
]

base = "in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31"

result = []
total_repos = 0
for group in keywords_list:
    or_query = " AND ".join(group)
    full_query = f"{or_query} {base}"
    
    url = "https://api.github.com/search/repositories"
    params = {
        "q": full_query,
        "per_page": 1  # just to get total_count
    }

    response = requests.get(url, headers=HEADERS, params=params)
    data = response.json()

    if response.status_code == 200:
        count = data["total_count"]
        total_repos += count
        print(f"쿼리 {full_query} → {count}개")
        repo_dic ={}
        repo_dic[group[0]] = count
        result.append(repo_dic)
    else:
        print("❌ 오류 발생:", data)
    time.sleep(3)

print(f"\n📦 전체 추정 합계: {total_repos}개")


쿼리 "ai" AND "agentic" in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 → 557개

📦 전체 추정 합계: 557개


In [129]:
'''quantum OR ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 15.1k
quantum AND ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 110
quantum in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 751
ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 14.5k'''
full_query = f'"ai agentic" in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31'
#'natural language processing','language model','nlp'],





url = "https://api.github.com/search/repositories"
params = {
      "q": full_query,
      "per_page": 1  # just to get total_count
}

response = requests.get(url, headers=HEADERS, params=params)
data = response.json()
data["total_count"]

500

In [135]:
'''quantum OR ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 15.1k
quantum AND ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 110
quantum in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 751
ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 14.5k'''
full_query = f'"computer vision" OR "multi-modal" OR "robotics" in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31'

url = "https://api.github.com/search/repositories"
params = {
      "q": full_query,
      "per_page": 1  # just to get total_count
}

response = requests.get(url, headers=HEADERS, params=params)
data = response.json()
data["total_count"]

9931

In [115]:
'''quantum OR ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 15.1k
quantum AND ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 110
quantum in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 751
ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31 -> 14.5k'''
full_query = f"quantum OR ai in:name,description,readme language:python created:2012-01-01..2024-12-31 stars:>10 pushed:2022-01-01..2024-12-31"

url = "https://api.github.com/search/repositories"
params = {
      "q": full_query,
      "per_page": 1  # just to get total_count
}

response = requests.get(url, headers=HEADERS, params=params)
data = response.json()
data["total_count"]

15118

In [69]:
total_keywords = pd.read_csv("keywords_2020~2024.csv")
rows = []

for re in result:
    for k, v in re.items():
        rows.append({'keyword': k, 'github_count': v})

github_keywords = pd.DataFrame(rows)
total_keywords = pd.merge(total_keywords, github_keywords, on='keyword')
total_keywords.to_csv("keywords_2020~2024(+github_search).csv")

In [6]:
from datasets import load_dataset

dataset = load_dataset("DeepDive-AI/AI-related-documents")
dataset = dataset['train']
dataset

README.md:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

entity_paper_ai.csv:   0%|          | 0.00/16.8G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/33 [00:00<?, ?it/s]

Dataset({
    features: ['id', 'eid', 'paper_title', 'source_title', 'paper_summary', 'paper_authors_name', 'paper_authors_id', 'pub_year', 'cited_num', 'author_keyword_json', 'doi', 'paper_type', 'language', 'affiliations_info'],
    num_rows: 9508512
})

In [40]:
# pub_year > 2012인 샘플만 필터링
filtered_dataset = dataset.filter(lambda example: example["pub_year"] >= 2020)

Filter:   0%|          | 0/9508512 [00:00<?, ? examples/s]

In [41]:
author_keywords = filtered_dataset["author_keyword_json"]
author_keywords

['[]',
 '[]',
 '[]',
 '[]',
 '["\\" Kansei\\"", "Force interaction", "Haptic interface", "Human-cooperative robot", "Human-machine interface"]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '["Animation", "Interactive visualization", "Reeb graph", "Skeleton"]',
 '[]',
 '["Analytic functions", "Hadamard product (or convolution)", "q-difference operator", "Subordinating factor sequence", "Univalent functions"]',
 '[]',
 '[]',
 '["Multi-criteria decision-making", "Organisation", "System dynamics", "Tourism"]',
 '["Content", "Teenagers", "Users\' language", "Web", "Writing"]',
 '[]',
 '["Animation", "Database", "Metadata", "Multimedia presentations", "Reuse"]',
 '[]',
 '["Distributed data mining", "Evolutionary algorithms", "Feature ranking", "Feature selection", "Multi-o

In [48]:
len(author_keywords)

1861229

In [42]:
import pandas as pd
import json
from collections import Counter



# 키워드 추출 및 정규화
all_keywords = []
i = 0
for entry in author_keywords:
    try:
        keywords = json.loads(entry)  # 문자열 → 리스트
        normalized = [kw.strip().lower() for kw in keywords]
        if normalized:
            i+=1
            all_keywords.extend(normalized)
    except json.JSONDecodeError:
        continue

# 빈도 계산
keyword_counts = Counter(all_keywords)
print(i)


1494252


In [ ]:
# 결과 출력
result = keyword_counts.most_common(300)
result

[('deep learning', 49314),
 ('machine learning', 48523),
 ('optimization', 10897),
 ('artificial intelligence', 10581),
 ('convolutional neural network', 9904),
 ('feature extraction', 9483),
 ('reinforcement learning', 8904),
 ('neural networks', 8492),
 ('internet of things', 8025),
 ('neural network', 7901),
 ('classification', 7850),
 ('task analysis', 6843),
 ('blockchain', 6688),
 ('security', 6002),
 ('object detection', 5906),
 ('computer vision', 5856),
 ('transfer learning', 5788),
 ('convolutional neural networks', 5731),
 ('cnn', 5543),
 ('training', 5461),
 ('attention mechanism', 5450),
 ('iot', 5232),
 ('federated learning', 4979),
 ('transformer', 4757),
 ('deep reinforcement learning', 4687),
 ('clustering', 4666),
 ('cloud computing', 4542),
 ('feature selection', 4535),
 ('natural language processing', 4394),
 ('edge computing', 4327),
 ('anomaly detection', 4291),
 ('image processing', 4246),
 ('fault diagnosis', 4210),
 ('genetic algorithm', 4164),
 ('covid-19', 39

In [43]:
result= pd.DataFrame(result,columns=['keyword','counts'])
result = result[result['counts']>1]
result

,keyword,counts
0,deep learning,49314
1,machine learning,48523
2,optimization,10897
3,artificial intelligence,10581
4,convolutional neural network,9904
...,...,...
295,cyber security,867
296,microgrids,866
297,speech recognition,860
298,cameras,859


In [44]:
result.to_csv("keywords_2020~2024.csv")